<img src="imagens/banner.png" alt="Módulo 02: Fundamentos da Web" width="1100">

# Módulo 02: Fundamentos da Web

Um scraper faz, por meio de código, o mesmo que o navegador faz quando você visita um site: pede uma página a um servidor, recebe o conteúdo e o interpreta. Para escrever um scraper, portanto, é preciso entender essa conversa: **quem fala com quem, em que formato e o que pode dar errado**.

Este módulo apresenta esses fundamentos, que serão usados em todos os módulos seguintes.

<img src="imagens/mapa-do-curso.png" alt="Mapa dos 10 módulos do curso, com o Módulo 02 destacado" width="1100">

## Neste módulo você vai

- Entender o modelo **cliente e servidor** e o papel do **DNS**;
- Reconhecer as partes de uma **URL** e perceber que o caminho funciona como o de uma pasta;
- Ver como é uma conversa **HTTP**: requisição, resposta, métodos, **códigos de status** e **cabeçalhos**;
- Ler a estrutura de uma página **HTML** e entendê-la como uma árvore, o **DOM**;
- Usar as **ferramentas do desenvolvedor** (DevTools) do navegador para inspecionar elementos e requisições;
- Conhecer o **robots.txt** e as boas práticas de uma coleta responsável.

No final, um **apêndice opcional** mostra como colocar no ar um servidor web no seu próprio computador. Ele não é necessário para os módulos seguintes.

**Tempo estimado:** de 1h a 1h30, mais cerca de 30 minutos para o apêndice.

**Como este módulo funciona:** como no Módulo 01, cada explicação curta é seguida de uma célula curta para executar. Os exercícios **Sua vez** são divididos em passos, cada um com a sua célula e a resposta oculta logo abaixo.

**Antes de começar:** abra o ambiente do curso seguindo a rotina do Módulo 01 (terminal na pasta do curso, `.venv` ativado e o Jupyter ou o VS Code aberto) e abra este notebook, `02-fundamentos-da-web.ipynb`.

**Navegador:** os exemplos usam o **Firefox** e o **Edge**, mas qualquer navegador moderno serve. O Firefox é o navegador de referência do curso: é de código aberto, mantido pela Mozilla, e usa um motor próprio, o **Gecko**, enquanto o Chrome e o Edge usam o **Chromium**, mantido pelo Google. Ele também será usado com o Selenium, no Módulo 09. Se ainda não o tiver, ele pode ser baixado em [firefox.com](https://www.firefox.com/pt-BR/).

---

# Parte 1: Como a web funciona

## 1.1 Cliente e servidor

Toda visita a um site envolve dois papéis:

- O **cliente** é quem pede: o navegador, ou um scraper.
- O **servidor** é um computador, ligado à internet o tempo todo, que guarda o site e responde aos pedidos.

Antes de fazer o pedido, o cliente precisa descobrir **onde** o servidor está. Na internet, cada computador é identificado por um número, o **endereço IP** (por exemplo, `35.211.122.109`). Como nomes são mais fáceis de lembrar do que números, existe o **DNS** (*Domain Name System*), um serviço que traduz o nome do site para o seu IP, como uma lista telefônica.

O diagrama mostra o caminho completo, do endereço digitado até a página na tela:

<img src="imagens/caminho-da-pagina.png" alt="Diagrama de sequência: o navegador consulta o DNS, recebe o IP, faz a requisição HTTP ao servidor, recebe o HTML e depois os arquivos extras" width="1100">

1. O navegador pergunta ao DNS qual é o IP de `books.toscrape.com`;
2. O DNS responde com o endereço;
3. O navegador envia ao servidor uma **requisição HTTP**, pedindo a página;
4. O servidor devolve uma **resposta**, que contém o **HTML** da página;
5. Lendo o HTML, o navegador descobre que precisa de mais arquivos (estilos, imagens, scripts) e faz um pedido para cada um;
6. Com tudo em mãos, o navegador monta a página na tela.

Um scraper faz os passos 1 a 4. A diferença está no final: em vez de exibir a página, ele **lê o HTML e extrai os dados**.

### Consultando o DNS pelo Python

O módulo `socket`, que já vem com o Python, permite fazer a mesma consulta que o navegador faz no passo 1.

▶️ A função `socket.gethostbyname()` recebe um nome e devolve o endereço IP correspondente. Consulte o IP do books.toscrape.com:

In [ ]:
import socket

socket.gethostbyname("books.toscrape.com")

▶️ Agora consulte três nomes de uma vez. O `for` repete a consulta para cada nome da lista:

In [ ]:
for nome in ["books.toscrape.com", "quotes.toscrape.com", "localhost"]:
    print(nome, "->", socket.gethostbyname(nome))

Aparecem três endereços. Repare em dois detalhes:

- `books.toscrape.com` e `quotes.toscrape.com` têm **o mesmo IP**: os dois sites ficam no mesmo servidor. Na seção 2.4 você vai ver como o servidor sabe qual dos sites está sendo pedido.
- `localhost` aponta para `127.0.0.1`, um endereço especial que significa **"este computador"**. Ele é usado no apêndice opcional, no final do módulo.

> 💡 O IP de um site pode mudar com o tempo, e por isso o seu resultado pode ser diferente do mostrado neste material.

## 1.2 Anatomia de uma URL

A **URL** (*Uniform Resource Locator*) é o endereço completo de um recurso na web. Cada parte tem uma função:

<img src="imagens/url-anatomia.png" alt="Duas URLs divididas em partes coloridas: protocolo, domínio, porta, caminho e parâmetros" width="1100">

- O **protocolo** define as regras da conversa. `https` é a versão criptografada do `http`.
- O **domínio** identifica o servidor. É o nome que o DNS transforma em IP.
- A **porta** indica qual programa, dentro do servidor, vai atender. Quando ela não aparece na URL, vale a porta padrão do protocolo: 443 para HTTPS e 80 para HTTP. O segundo exemplo da imagem, com a porta `8000`, é o do apêndice opcional.
- O **caminho** indica qual recurso, dentro do servidor, está sendo pedido.
- Os **parâmetros** (*query string*) vêm depois do `?`, no formato `nome=valor`, separados por `&`. Eles ajustam o pedido: qual página de resultados, qual termo de busca, qual filtro. No Módulo 05, alterar o parâmetro `page` será a forma de percorrer todas as páginas de um catálogo.

### Desmontando uma URL com Python

A função `urlsplit()`, do módulo `urllib.parse`, separa uma URL nas partes do diagrama.

▶️ Separe as partes de uma URL da loja de exemplo do webscraper.io. O resultado guarda cada parte com um nome:

In [ ]:
from urllib.parse import urlsplit, parse_qs

partes = urlsplit("https://webscraper.io/test-sites/e-commerce/static/computers/tablets?page=2")
partes

▶️ Cada parte é acessada com um ponto e o nome dela. O domínio fica em `hostname`:

In [ ]:
partes.hostname

▶️ O caminho fica em `path`:

In [ ]:
partes.path

▶️ A porta fica em `port`. Use `print()`, porque o valor é `None`:

In [ ]:
print(partes.port)

A porta não foi escrita na URL, e por isso vale a porta padrão do `https`, a 443.

▶️ Os parâmetros ficam em `query`, como texto. A função `parse_qs()` os transforma em um dicionário:

In [ ]:
parse_qs(partes.query)

O valor de `page` vem dentro de uma lista porque um mesmo parâmetro pode aparecer mais de uma vez na URL.

### Sua vez: desmontando uma URL

**Passo 1 de 3.** A página de times de hóquei do scrapethissite tem uma busca. Separe as partes da URL `https://www.scrapethissite.com/pages/forms/?q=bruins&page_num=1` com o `urlsplit()` e guarde em `busca`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
busca = urlsplit("https://www.scrapethissite.com/pages/forms/?q=bruins&page_num=1")
```
</details>

**Passo 2 de 3.** Mostre o caminho da URL.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
busca.path
```
</details>

**Passo 3 de 3.** Transforme os parâmetros em um dicionário. Depois, abra a URL no navegador e troque o valor de `q` por `rangers`: o que muda na página?

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
parse_qs(busca.query)
```
</details>

O parâmetro `q` é o termo da busca, e `page_num` é a página dos resultados. Com `q=rangers`, a tabela mostra só os times com esse nome.

## 1.3 Uma URL é o caminho até um arquivo

O caminho de uma URL foi inspirado na forma como os arquivos são organizados em pastas. Em sites simples, essa relação é literal: o servidor guarda os arquivos das páginas em pastas, e o caminho da URL aponta para um deles. O books.toscrape.com é um site assim:

<img src="imagens/url-arquivos.png" alt="À esquerda, a organização dos arquivos do books.toscrape.com em pastas; à direita, o endereço de cada um; a página 51 do catálogo não existe e gera erro 404" width="1120">

Sites grandes não costumam guardar cada página como um arquivo: elas são geradas por programas no momento do pedido, a partir de bancos de dados. Mesmo assim, a ideia de **caminho** continua a mesma, e é com ela que um scraper navega por um site.

> 💡 No apêndice opcional, no final do módulo, você pode transformar uma pasta do seu próprio computador em um site e ver essa relação acontecer.

> **Referências da Parte 1:** [Como a web funciona](https://developer.mozilla.org/pt-BR/docs/Learn_web_development/Getting_started/Web_standards/How_the_web_works) (MDN); [What is a URL?](https://developer.mozilla.org/en-US/docs/Learn_web_development/Howto/Web_mechanics/What_is_a_URL) e [What is a domain name?](https://developer.mozilla.org/en-US/docs/Learn_web_development/Howto/Web_mechanics/What_is_a_domain_name) (MDN, em inglês); [urllib.parse](https://docs.python.org/pt-br/3/library/urllib.parse.html) e [socket](https://docs.python.org/pt-br/3/library/socket.html) (documentação do Python).

---

# Parte 2: HTTP, a conversa entre cliente e servidor

## 2.1 Requisição e resposta

**HTTP** (*HyperText Transfer Protocol*) é o protocolo, ou seja, o conjunto de regras que cliente e servidor seguem para conversar. Uma característica importante é que **a conversa é feita em texto**, que pode ser lido por pessoas. Cada pedido é uma **requisição** (*request*), e cada retorno é uma **resposta** (*response*), e as duas mensagens têm a mesma estrutura:

<img src="imagens/http-mensagens.png" alt="Uma requisição GET para a página 2 do books.toscrape.com e a resposta 200 OK lado a lado, cada uma dividida em linha inicial, cabeçalhos, linha em branco e corpo" width="1120">

- A **linha inicial** resume a mensagem. Na requisição, ela traz o método, o caminho e a versão do protocolo; na resposta, a versão, o código de status e a descrição.
- Os **cabeçalhos** (*headers*) trazem informações sobre a mensagem, no formato `Nome: valor`.
- Uma **linha em branco** marca o fim dos cabeçalhos.
- O **corpo** (*body*) traz o conteúdo. Em uma resposta, é o HTML da página.

### Escrevendo uma requisição à mão

Para mostrar que o HTTP é só texto, as células abaixo conversam com o servidor do books.toscrape.com diretamente, pelo módulo `socket`, sem nenhuma biblioteca de scraping. Não é preciso decorar esse código: no Módulo 03, a biblioteca `requests` faz tudo isso em uma linha. O objetivo aqui é ver a conversa sem intermediários.

▶️ Monte o texto da requisição, linha por linha. No HTTP, as linhas terminam com `\r\n` (os caracteres de "retorno" e "nova linha"), e uma linha vazia no final marca o fim do pedido. O `print()` mostra o texto como ele será enviado:

In [ ]:
requisicao = (
    "GET /catalogue/page-2.html HTTP/1.1\r\n"
    "Host: books.toscrape.com\r\n"
    "Connection: close\r\n"
    "\r\n"
)
print(requisicao)

É exatamente o formato da requisição do diagrama: a linha inicial, os cabeçalhos e a linha em branco.

▶️ Abra uma conexão com o servidor na **porta 80**, a do HTTP sem criptografia, envie o texto e receba a resposta. O `recv()` lê a resposta em pedaços, e o laço junta os pedaços até o servidor encerrar a conexão:

In [ ]:
with socket.create_connection(("books.toscrape.com", 80)) as conexao:
    conexao.sendall(requisicao.encode("utf-8"))
    recebido = b""
    while True:
        pedaco = conexao.recv(4096)
        if not pedaco:
            break
        recebido += pedaco
len(recebido)

O número é o tamanho da resposta, em bytes.

▶️ Mostre os primeiros 700 caracteres da resposta, como texto:

In [ ]:
print(recebido.decode("utf-8")[:700])

A resposta aparece como texto: a linha `HTTP/1.1 200 OK`, os cabeçalhos (`Date`, `Content-Type`, `Content-Length`...), uma linha em branco e, por fim, o começo do HTML da página 2 do catálogo. As linhas que começam com `<!--[if` são instruções para versões muito antigas do Internet Explorer, e podem ser ignoradas.

> 💡 Mostrar o HTTP conversando com o servidor diretamente, por um *socket*, é o caminho usado pelo Dr. Charles Severance no início do curso *Django for Everybody*, uma das referências deste módulo (veja a seção Referências).

## 2.2 Métodos: o que o cliente quer fazer

A primeira palavra da requisição é o **método**, que diz que tipo de ação o cliente quer realizar. Os dois mais comuns são:

| Método | Para que serve | Exemplo |
|---|---|---|
| `GET` | **Buscar** um recurso | Abrir uma página, clicar em um link, carregar uma imagem |
| `POST` | **Enviar** dados para o servidor | Enviar um formulário, fazer login, publicar um comentário |

A grande maioria das requisições de um scraper é `GET`, já que o objetivo é **ler** páginas. O `POST` aparece quando é preciso preencher um formulário para chegar aos dados. Existem outros métodos, como `PUT` e `DELETE`, usados principalmente por sistemas que conversam entre si, e eles raramente aparecem no scraping.

> 💡 Os parâmetros de uma requisição `GET` ficam na própria URL, depois do `?`, como você viu na seção 1.2. Em um `POST`, os dados vão no corpo da requisição.

## 2.3 Códigos de status: como foi o pedido

Toda resposta traz um **código de status** de três dígitos. O primeiro dígito indica a família do resultado:

<img src="imagens/status-codes.png" alt="As quatro famílias de códigos de status: 2xx sucesso, 3xx redirecionamento, 4xx erro do cliente e 5xx erro do servidor, com exemplos e o que o scraper deve fazer em cada caso" width="1100">

Para um scraper, conferir o código de status é o primeiro passo depois de cada pedido: só faz sentido extrair dados de uma resposta `200`.

### Provocando códigos diferentes

A partir daqui, os pedidos são feitos com a biblioteca `requests`, instalada no Módulo 01. Ela será estudada em detalhes no Módulo 03; neste módulo, ela serve para observar a conversa. A função `requests.get()` faz um pedido `GET` e devolve a resposta, que tem os atributos `status_code` (o código) e `reason` (a descrição).

▶️ Peça a página 2 do catálogo, que existe:

In [ ]:
import requests

resposta = requests.get("https://books.toscrape.com/catalogue/page-2.html", timeout=20)
print(resposta.status_code, resposta.reason)

▶️ Agora a página 51. O catálogo tem só 50 páginas:

In [ ]:
resposta = requests.get("https://books.toscrape.com/catalogue/page-51.html", timeout=20)
print(resposta.status_code, resposta.reason)

▶️ Por fim, um endereço do scrapethissite **sem** a barra no final. O argumento `allow_redirects=False` faz o `requests` mostrar o redirecionamento em vez de segui-lo automaticamente:

In [ ]:
resposta = requests.get("https://www.scrapethissite.com/pages/simple", allow_redirects=False, timeout=20)
print(resposta.status_code, resposta.reason)

O servidor respondeu `308`, um redirecionamento. Para onde ele manda o cliente fica no cabeçalho `Location` (seção 2.4).

▶️ Veja o endereço indicado:

In [ ]:
resposta.headers["Location"]

O endereço correto é `/pages/simple/`, com a barra. Sem o `allow_redirects=False`, o `requests` iria para ele sozinho, como faz o navegador.

> 💡 A descrição que acompanha o código pode variar de um servidor para outro (`Not Found`, `NOT FOUND`, `File not found`). O que importa é o número.

### Sua vez: prevendo códigos de status

**Passo 1 de 3.** O quotes.toscrape.com divide as citações em páginas: `/page/1/`, `/page/2/` e assim por diante. **Antes de executar**, anote qual código você espera para `https://quotes.toscrape.com/page/10/` e confira com um pedido.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
resposta = requests.get("https://quotes.toscrape.com/page/10/", timeout=20)
resposta.status_code
```
</details>

**Passo 2 de 3.** Faça o mesmo com a página 11, `https://quotes.toscrape.com/page/11/`. Depois, abra-a no navegador.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
resposta = requests.get("https://quotes.toscrape.com/page/11/", timeout=20)
resposta.status_code
```
</details>

**Passo 3 de 3.** E com a página 10 **sem** a barra final, `https://quotes.toscrape.com/page/10`, usando `allow_redirects=False`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
resposta = requests.get("https://quotes.toscrape.com/page/10", allow_redirects=False, timeout=20)
resposta.status_code
```
</details>

- A página 10 dá `200`: é a última com citações.
- A página 11 **também dá `200`**, mas mostra apenas *No quotes found!*. O site não usa o `404` para páginas vazias.
- Sem a barra final, o resultado é `308`: um redirecionamento para o endereço com a barra.

O caso da página 11 é uma lição importante para o Módulo 05: **um código `200` não garante que a página tem conteúdo**.

## 2.4 Cabeçalhos: informações sobre a mensagem

Os cabeçalhos levam informações extras nos dois sentidos da conversa. Alguns que vão aparecer ao longo do curso:

| Cabeçalho | Enviado por | Para que serve |
|---|---|---|
| `Host` | cliente | Qual site está sendo pedido. É assim que um mesmo servidor atende vários sites, como os dois sites de treino que tinham o mesmo IP na seção 1.1 |
| `User-Agent` | cliente | Identifica o programa que está fazendo o pedido |
| `Accept` | cliente | Que tipos de conteúdo o cliente aceita receber |
| `Content-Type` | servidor | O tipo do conteúdo do corpo: `text/html`, `application/json`, `image/jpeg`... |
| `Content-Length` | servidor | O tamanho do corpo, em bytes |
| `Last-Modified` | servidor | Quando o arquivo foi modificado pela última vez |
| `Location` | servidor | Para onde ir, em um redirecionamento (`3xx`) |

O `requests` guarda os cabeçalhos dos dois lados: os enviados ficam em `resposta.request.headers`, e os recebidos, em `resposta.headers`.

▶️ Peça a página inicial do books.toscrape.com:

In [ ]:
resposta = requests.get("https://books.toscrape.com/", timeout=20)

▶️ Veja os cabeçalhos que o `requests` **enviou**:

In [ ]:
for nome, valor in resposta.request.headers.items():
    print(nome, ":", valor)

▶️ E os que o servidor **devolveu**:

In [ ]:
for nome, valor in resposta.headers.items():
    print(nome, ":", valor)

Repare em três cabeçalhos:

- O `User-Agent` enviado é `python-requests/...`: o requests avisa a qualquer site que o pedido vem de um programa, e não de um navegador. Alguns sites recusam pedidos com esse tipo de identificação; no Módulo 03 você vai ver como e quando alterá-lo.
- O `Last-Modified` recebido mostra uma data de 2023: a página inicial é um arquivo pronto, que não muda desde então, como você viu na seção 1.3.
- O `Content-Type` recebido é `text/html`, **sem informar a codificação** do texto (o `charset`, como `utf-8`). Quando essa informação falta, o `requests` adivinha, e pode errar.

▶️ Veja o efeito dessa adivinhação no preço do primeiro livro. O método `find()` devolve a posição do trecho `price_color` no HTML recebido, e a célula mostra os 25 caracteres a partir dela:

In [ ]:
posicao = resposta.text.find("price_color")
resposta.text[posicao:posicao + 25]

O preço aparece como `Â£51.77`, e não como `£51.77`: o `requests` leu o texto com a codificação errada. Esse problema é comum em sites reais, e a forma de resolvê-lo é um dos assuntos do Módulo 03.

## 2.5 HTTPS

O **HTTPS** é o HTTP dentro de um canal **criptografado**. A conversa tem as mesmas mensagens que você viu acima, mas ninguém no caminho entre o cliente e o servidor (a rede Wi-Fi, o provedor de internet) consegue lê-las ou alterá-las. O cadeado ao lado do endereço, no navegador, indica uma conexão HTTPS.

Para quem escreve um scraper, a diferença é pequena: basta usar `https://` na URL, e o `requests` cuida da criptografia. A requisição feita à mão na seção 2.1 usou a porta 80, do HTTP sem criptografia, justamente para que o texto da conversa pudesse ser lido sem esse passo.

> **Referências da Parte 2:** [Uma visão geral do HTTP](https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Guides/Overview), [Mensagens HTTP](https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Guides/Messages), [Métodos de requisição](https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Reference/Methods), [Códigos de status](https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Reference/Status) e [Cabeçalhos HTTP](https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Reference/Headers) (MDN); [RFC 9110: HTTP Semantics](https://www.rfc-editor.org/rfc/rfc9110), a especificação oficial do HTTP (em inglês); [Django for Everybody](https://www.youtube.com/watch?v=o0XbHvKxw7Y), de Charles Severance, no canal freeCodeCamp.org: o início do curso trata do ciclo de requisição e resposta e de *sockets* (em inglês).

---

# Parte 3: HTML e DOM

## 3.1 O que é HTML

O corpo das respostas que você viu até aqui está escrito em **HTML** (*HyperText Markup Language*), a linguagem que descreve a **estrutura** de uma página: o que é título, o que é parágrafo, o que é link, o que é tabela. O HTML não é uma linguagem de programação; ele **marca** o texto com etiquetas, as *tags*.

Para os exemplos desta parte, a pasta deste módulo traz um site pequeno, feito para o curso: a **Livraria Exemplo**, na pasta `site-exemplo`. Abra o arquivo `site-exemplo/index.html` com dois cliques: o navegador o mostra direto do disco, sem precisar de servidor.

<img src="imagens/html-anatomia.png" alt="Um elemento a da Livraria Exemplo dividido em tag de abertura, atributo class, atributo href, conteúdo e tag de fechamento; abaixo, um elemento li com outros elementos dentro" width="1100">

Um **elemento** é formado pela tag de abertura, pelo conteúdo e pela tag de fechamento. Os **atributos** ficam dentro da tag de abertura e trazem informações adicionais. Para o scraping, dois atributos são especialmente importantes:

- `class` é um rótulo, usado pelo site para aplicar estilos. Elementos com o mesmo papel costumam ter a mesma classe: todos os preços com `class="preco"`, por exemplo. **É por esses rótulos que um scraper encontra os dados.**
- `href` (em links) e `src` (em imagens) guardam **endereços**, que um scraper usa para navegar até outras páginas.

As tags mais frequentes:

| Tag | Uso |
|---|---|
| `html`, `head`, `body` | A estrutura geral: o documento, as informações sobre ele e o conteúdo visível |
| `title` | O título que aparece na aba do navegador |
| `h1` a `h6` | Títulos, do maior para o menor |
| `p` | Parágrafo |
| `a` | Link (o endereço fica no `href`) |
| `img` | Imagem (o endereço fica no `src`) |
| `ul`, `ol`, `li` | Listas e seus itens |
| `table`, `tr`, `th`, `td` | Tabela, linha, célula de cabeçalho e célula de dados |
| `div`, `span` | Contêineres genéricos, usados para agrupar e rotular partes da página |

> 💡 A apresentação de HTML desta parte segue a linha da aula de HTML do curso *Django for Everybody*, do Dr. Charles Severance, publicado no canal freeCodeCamp.org. A aula começa em [1:34:43 do vídeo](https://www.youtube.com/watch?v=o0XbHvKxw7Y&t=5683s) (em inglês) e é um bom complemento a esta seção.

### O HTML da Livraria Exemplo

▶️ Leia o arquivo `index.html` da pasta `site-exemplo`. O `Path` representa o caminho do arquivo, e o `read_text()` lê o conteúdo como texto:

In [ ]:
from pathlib import Path

html = Path("site-exemplo/index.html").read_text(encoding="utf-8")
len(html)

O arquivo tem pouco mais de mil caracteres.

▶️ Mostre o texto inteiro:

In [ ]:
print(html)

Compare o texto com a página que você abriu no navegador: cada livro é um item de lista (`li`) com a classe `livro`, e dentro dele estão o título (`a`), o autor e o preço (`span`), cada um com a sua classe.

> ⚠️ Se aparecer `FileNotFoundError`, o notebook não está sendo executado a partir da pasta `02-fundamentos-da-web`. Confira se você abriu o arquivo `02-fundamentos-da-web.ipynb` que está dentro dessa pasta.

## 3.2 O DOM: a página como uma árvore

Como os elementos ficam dentro de outros elementos, o HTML forma uma **árvore**. Quando o navegador lê o HTML, ele monta essa árvore na memória, e é ela que se chama **DOM** (*Document Object Model*). Para descrever as posições na árvore, usa-se um vocabulário de família:

<img src="imagens/dom-arvore.png" alt="A árvore do DOM da página inicial da Livraria Exemplo, com o pai, os filhos e os irmãos destacados em cores" width="1100">

Esse vocabulário será a base do Módulo 04, em que você vai aprender a escrever caminhos pela árvore para chegar a um dado específico.

### Por que não basta procurar no texto

Como o HTML é texto, é tentador procurar os dados com as ferramentas de texto do Python.

▶️ O método `count()` conta quantas vezes um trecho aparece no texto. Conte os livros:

In [ ]:
html.count('class="livro"')

▶️ E os preços:

In [ ]:
html.count('class="preco"')

O resultado é 4 e 4, o que está certo. Mas contar é fácil; **extrair** cada preço junto do título certo, em uma página com centenas de elementos, é outra história. Para isso, existem ferramentas que entendem a árvore, como a biblioteca `parsel`, que é o assunto do Módulo 04.

## 3.3 Inspecionando a página com o DevTools

Todo navegador moderno tem as **ferramentas do desenvolvedor** (*DevTools*), que mostram o DOM, as requisições e muito mais. Para quem faz scraping, elas são indispensáveis: é com elas que você descobre **onde** está o dado que quer extrair.

Para abrir o DevTools no Firefox, no Edge ou no Chrome:

- Clique com o **botão direito** sobre qualquer parte da página e escolha **Inspecionar**; ou
- Aperte **`F12`** ou **`Ctrl + Shift + I`** (no macOS, **`Cmd + Option + I`**).

> 💡 Em alguns notebooks, as teclas `F1` a `F12` só funcionam junto com a tecla `Fn`.

### O painel de elementos

Abra **[books.toscrape.com](https://books.toscrape.com)**, clique com o botão direito sobre o preço do primeiro livro e escolha **Inspecionar**. A imagem mostra o resultado no Edge:

<img src="imagens/devtools-elementos.png" alt="O site books.toscrape.com com o preço do primeiro livro destacado e, abaixo, o DevTools do Edge com a aba Elementos aberta e o elemento p com a classe price_color selecionado" width="1060">

- ① O elemento inspecionado: o preço do primeiro livro.
- ② O mesmo elemento no DOM, selecionado no painel: `<p class="price_color">£51.77</p>`.
- ③ O painel que mostra a árvore. No Edge e no Chrome, ele se chama **Elementos** (*Elements*); no Firefox, **Inspetor** (*Inspector*). As setas ▶ abrem e fecham os ramos.
- ④ A classe do elemento, `price_color`. É esse rótulo que um scraper vai usar para encontrar todos os preços da página.

Repare também na barra na parte de baixo do painel: ela mostra o caminho do elemento desde a raiz da árvore (`... article.product_pod > div.product_price > p.price_color`), ou seja, os seus **ancestrais**.

> 💡 Passe o mouse sobre as linhas do painel: a parte correspondente da página fica destacada. É um jeito rápido de descobrir qual elemento contém o quê.

### Posicionando o painel: à direita, embaixo ou em uma janela própria

O DevTools pode ficar encaixado **embaixo**, à **direita** ou à **esquerda** da página, ou ainda em uma **janela separada**. Para o trabalho de scraping, a posição **à direita** costuma ser a mais confortável: a página continua visível quase por inteiro e pode ser rolada com facilidade enquanto você inspeciona, ao passo que, com o painel embaixo, sobra pouca altura para ela. A imagem mostra o mesmo preço inspecionado no Firefox, com o painel à direita:

<img src="imagens/devtools-firefox.png" alt="O Firefox com o DevTools encaixado à direita, o preço do primeiro livro selecionado no Inspector e, ao lado, as opções do menu de posicionamento do painel" width="1152">

- ① O preço, inspecionado na página, que continua visível e pode ser rolada.
- ② A linha correspondente, selecionada no **Inspector**.
- ③ A aba **Inspector**, o nome do painel de elementos no Firefox.
- ④ O menu **···** do DevTools, onde se escolhe a posição do painel: *Dock to Bottom* (embaixo), *Dock to Right* (à direita), *Dock to Left* (à esquerda) ou *Separate Window* (janela própria).

No Edge e no Chrome, as mesmas opções ficam no menu **⋮** do DevTools, em uma linha de ícones de encaixe. Nesses navegadores, o atalho **`Ctrl + Shift + D`** alterna entre a posição atual e a última usada.

> 💡 A janela separada é útil para quem trabalha com dois monitores: a página fica em uma tela, e o DevTools na outra.

### O painel de rede

O painel **Rede** (*Network*, no Firefox e na versão em inglês dos outros navegadores) registra todas as requisições que a página faz. Com ele aberto, recarregue a página (`F5`). A imagem mostra o resultado no Edge:

<img src="imagens/devtools-rede.png" alt="A aba Rede do DevTools com a lista de requisições: a primeira é o documento books.toscrape.com, com status 200 e tipo document; na parte de baixo, o resumo com 32 solicitações" width="1000">

- ① A aba **Rede**.
- ② A primeira requisição é o **documento HTML**, com o status `200` e o tipo `document`. É essa resposta que interessa a um scraper.
- ③ O resumo: para montar uma única página, o navegador fez **32 requisições** (o HTML, as folhas de estilo, os scripts, as imagens das capas), como no passo 5 do diagrama da seção 1.1.

Clicando em uma requisição, o DevTools mostra os cabeçalhos enviados e recebidos, e a resposta completa. No Módulo 07, este painel será usado para descobrir de onde vêm os dados de páginas que não os trazem no HTML.

> 💡 Se a lista aparecer vazia, é porque o painel foi aberto depois de a página carregar. Recarregue a página com o DevTools aberto.

### Sua vez: investigando com o DevTools

Em **[books.toscrape.com](https://books.toscrape.com)**, com o DevTools encaixado à direita, inspecione o **título** do primeiro livro.

1. Qual é a tag do elemento que contém o título? Ela está dentro de qual outra tag?
2. O texto que aparece na tela é `A Light in the ...`. Onde, no elemento, está o título **completo**?

<details>
<summary>Ver a resposta</summary>

O título é um link, `<a>`, dentro de um título de nível 3, `<h3>`:

```html
<h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>
```

O texto visível foi encurtado, mas o título completo está no atributo `title`. É comum que o dado mais útil esteja em um **atributo**, e não no texto visível: por isso, no Módulo 04, você vai aprender a extrair os dois.
</details>

## 3.4 O HTML recebido e o DOM nem sempre são iguais

Até aqui, o HTML que o servidor envia e o DOM que o navegador mostra eram a mesma coisa. Mas muitas páginas trazem programas em **JavaScript**, que rodam no navegador e podem **alterar o DOM depois** que a página chega: acrescentar elementos, buscar mais dados, montar listas inteiras.

O site de treino quotes.toscrape.com tem duas versões da mesma página, que parecem idênticas no navegador:

<img src="imagens/js-conteudo.png" alt="Comparação: o HTML de quotes.toscrape.com já traz as citações; o HTML de quotes.toscrape.com/js traz um script que cria as citações no navegador" width="1120">

Para ver o HTML **exatamente como o servidor o enviou**, use **Exibir código-fonte** (`Ctrl + U`), e não o painel de elementos, que mostra o DOM já alterado pelo JavaScript.

▶️ Baixe a versão normal e conte os blocos de citação no HTML recebido:

In [ ]:
normal = requests.get("https://quotes.toscrape.com/", timeout=20).text
normal.count('class="quote"')

▶️ Agora a versão com JavaScript:

In [ ]:
com_js = requests.get("https://quotes.toscrape.com/js/", timeout=20).text
com_js.count('class="quote"')

A primeira versão tem 10 blocos; a segunda, **nenhum**. O `requests` baixa o HTML, mas **não executa JavaScript**, e por isso enxerga a página como ela chega do servidor.

Esse é um dos limites mais importantes do scraping com `requests`. Há duas formas de contorná-lo, que serão vistas mais adiante no curso: buscar os dados direto na fonte de onde o JavaScript os tira (Módulo 07), ou controlar um navegador de verdade, que executa o JavaScript (Módulo 09).

> **Referências da Parte 3:** [Django for Everybody](https://www.youtube.com/watch?v=o0XbHvKxw7Y&t=5683s), de Charles Severance, no canal freeCodeCamp.org, aula de HTML a partir de 1:34:43 (em inglês); [HTML básico](https://developer.mozilla.org/pt-BR/docs/Learn_web_development/Getting_started/Your_first_website/Creating_the_content), [Iniciando com HTML](https://developer.mozilla.org/pt-BR/docs/Learn_web_development/Core/Structuring_content/Basic_HTML_syntax) e [Modelo de Objeto de Documento (DOM)](https://developer.mozilla.org/pt-BR/docs/Web/API/Document_Object_Model) (MDN); documentação do DevTools do [Firefox](https://firefox-source-docs.mozilla.org/devtools-user/) ([Inspector](https://firefox-source-docs.mozilla.org/devtools-user/page_inspector/), [Network Monitor](https://firefox-source-docs.mozilla.org/devtools-user/network_monitor/) e [posições do painel](https://firefox-source-docs.mozilla.org/devtools-user/tools_toolbox/), em inglês), do [Edge](https://learn.microsoft.com/pt-br/microsoft-edge/devtools/overview) e do [Chrome](https://developer.chrome.com/docs/devtools?hl=pt-br).

---

# Parte 4: Boas práticas

## 4.1 O arquivo robots.txt

Muitos sites publicam, na raiz do domínio, um arquivo chamado `robots.txt`, com regras para programas que acessam o site automaticamente (os "robôs", como os de buscadores e os scrapers). Ele indica quais caminhos não devem ser acessados e, às vezes, qual intervalo manter entre os pedidos.

<img src="imagens/robots-txt.png" alt="O robots.txt do webscraper.io linha por linha, com User-agent, Disallow e Sitemap explicados, e o Crawl-delay da Livraria Exemplo" width="1100">

O robots.txt é um **pedido** do dono do site, e não uma trava técnica: nada impede um programa de ignorá-lo. Respeitá-lo é uma questão de boa conduta e, em muitos casos, uma condição dos termos de uso do site.

### Consultando o robots.txt com Python

O módulo `urllib.robotparser`, que vem com o Python, interpreta o robots.txt e responde se um endereço pode ser acessado.

▶️ Crie uma função que baixa o robots.txt de um site com o `requests` e o entrega ao interpretador. Se o site não tiver robots.txt (resposta `404`), não há regras declaradas:

In [ ]:
from urllib.robotparser import RobotFileParser

def ler_robots(site):
    regras = RobotFileParser()
    resposta = requests.get(site + "/robots.txt", timeout=20)
    if resposta.status_code == 200:
        regras.parse(resposta.text.splitlines())
    else:
        regras.parse([])  # sem robots.txt: nenhuma regra declarada
    return regras

▶️ Leia as regras do webscraper.io:

In [ ]:
regras = ler_robots("https://webscraper.io")

▶️ O método `can_fetch()` responde se um endereço pode ser acessado. O `"*"` quer dizer "qualquer robô". Pergunte pela loja de exemplo:

In [ ]:
regras.can_fetch("*", "https://webscraper.io/test-sites/e-commerce/static/computers/tablets")

▶️ E por um caminho que está em um `Disallow`:

In [ ]:
regras.can_fetch("*", "https://webscraper.io/test-sites/pagination/")

A loja pode ser acessada (`True`), e `/test-sites/pagination/`, não (`False`).

O interpretador também lê o intervalo pedido entre as requisições (`Crawl-delay`). A Livraria Exemplo tem um robots.txt na pasta `site-exemplo`.

▶️ Leia esse arquivo direto do disco:

In [ ]:
regras_livraria = RobotFileParser()
regras_livraria.parse(Path("site-exemplo/robots.txt").read_text(encoding="utf-8").splitlines())
regras_livraria.crawl_delay("*")

▶️ Confira dois caminhos da Livraria:

In [ ]:
print("/livros/:   ", regras_livraria.can_fetch("*", "/livros/"))
print("/rascunhos/:", regras_livraria.can_fetch("*", "/rascunhos/"))

O intervalo pedido é de 2 segundos; `/livros/` pode ser acessado e `/rascunhos/`, não.

> 💡 O `RobotFileParser` também consegue baixar o arquivo sozinho, com o método `read()`. Mas ele se identifica como `Python-urllib`, e alguns sites, como o scrapethissite.com, recusam esse tipo de pedido. Nesse caso, o interpretador conclui que **nada** pode ser acessado. Baixar com o `requests` evita o problema.

### Sua vez: consultando um robots.txt

**Passo 1 de 3.** Use a função `ler_robots()` para ler as regras de `https://www.scrapethissite.com` e guarde em `regras_scrape`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
regras_scrape = ler_robots("https://www.scrapethissite.com")
```
</details>

**Passo 2 de 3.** Verifique se a página `https://www.scrapethissite.com/pages/simple/` pode ser acessada.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
regras_scrape.can_fetch("*", "https://www.scrapethissite.com/pages/simple/")
```
</details>

**Passo 3 de 3.** E a página `https://www.scrapethissite.com/faq/`?

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
regras_scrape.can_fetch("*", "https://www.scrapethissite.com/faq/")
```
</details>

As páginas de prática (`/pages/`) podem ser acessadas; `/faq/` e `/lessons/` estão em um `Disallow`.

## 4.2 Coleta responsável

O fato de uma informação estar visível no navegador não significa que qualquer uso dela seja permitido. Antes de escrever um scraper, vale responder a estas perguntas:

<img src="imagens/checklist-etica.png" alt="Oito perguntas para fazer antes de raspar um site: API oficial, termos de uso, robots.txt, ritmo, dados pessoais, barreiras, uso dos dados e identificação" width="1100">

Em resumo: colete apenas o necessário, em um ritmo que não sobrecarregue o site, respeitando as regras declaradas e a privacidade das pessoas. No Brasil, dados pessoais são protegidos pela **LGPD** (Lei Geral de Proteção de Dados, Lei nº 13.709/2018), inclusive quando estão publicados na internet.

Os sites de treino usados no curso (books.toscrape.com, quotes.toscrape.com, scrapethissite.com e as páginas de teste do webscraper.io) foram criados justamente para a prática de scraping, e as páginas usadas nos exercícios podem ser coletadas.

> **Referências da Parte 4:** [Introdução ao robots.txt](https://developers.google.com/search/docs/crawling-indexing/robots/intro?hl=pt-br) (Google); [RFC 9309: Robots Exclusion Protocol](https://www.rfc-editor.org/rfc/rfc9309), a especificação do robots.txt (em inglês); [urllib.robotparser](https://docs.python.org/pt-br/3/library/urllib.robotparser.html) (documentação do Python); [Lei nº 13.709/2018 (LGPD)](https://www.planalto.gov.br/ccivil_03/_ato2015-2018/2018/lei/l13709.htm) e [Autoridade Nacional de Proteção de Dados](https://www.gov.br/anpd/pt-br).

---

## Problemas comuns e como resolver

| Sintoma | Causa provável | Solução |
|---|---|---|
| `ConnectionError`, `Failed to establish a new connection` ou `Timeout` ao acessar um site | Sem conexão com a internet, ou o site está fora do ar | Confira a conexão e tente de novo. Se persistir, abra o endereço no navegador para ver se o site responde |
| `FileNotFoundError: ... site-exemplo/index.html` | O notebook não está rodando na pasta do módulo | Abra o notebook que está dentro de `02-fundamentos-da-web` |
| O navegador mostra uma versão antiga da página | O navegador guardou a página em cache | Recarregue com `Ctrl + F5` (no Firefox, também `Ctrl + Shift + R`) |
| O painel de rede está vazio | O DevTools foi aberto depois de a página carregar | Recarregue a página com o DevTools aberto |
| `F12` não abre o DevTools | Em alguns notebooks, as teclas de função dependem da tecla `Fn` | Use `Fn + F12`, `Ctrl + Shift + I` ou o botão direito → **Inspecionar** |
| Os nomes dos painéis estão diferentes | Cada navegador e idioma usa nomes próprios | Elementos = *Elements* (Edge, Chrome) = *Inspector* (Firefox); Rede = *Network*; *Console* é igual em todos |

---

# Retomada do Módulo 02

Resumo do que foi visto neste módulo:

<img src="imagens/retomada.png" alt="Mapa de revisão do módulo: cliente e servidor, URL, HTTP, códigos de status, HTML e DOM, DevTools e boas práticas" width="1100">

### O que você aprendeu

**Como a web funciona**

- O **cliente** pede e o **servidor** responde. O **DNS** traduz o nome do site para o seu endereço **IP**, e `localhost` é o próprio computador.
- Uma **URL** tem protocolo, domínio, porta, caminho e parâmetros. O caminho funciona como o de uma pasta, e um caminho sem recurso correspondente dá **404**.

**HTTP**

- A conversa é feita em **texto**: requisição e resposta têm linha inicial, cabeçalhos, linha em branco e corpo.
- O método `GET` busca recursos, e o `POST` envia dados.
- O **código de status** diz como foi o pedido: `2xx` sucesso, `3xx` redirecionamento, `4xx` erro no pedido e `5xx` erro no servidor.
- Os **cabeçalhos** trazem informações como o `User-Agent` de quem pede e o `Content-Type` do que foi enviado. Sem o `charset`, os acentos e símbolos podem sair trocados.

**HTML e DOM**

- O HTML marca o conteúdo com **tags**; os **atributos** `class`, `href` e `src` são os mais úteis para o scraping.
- O navegador transforma o HTML em uma árvore, o **DOM**, descrita com o vocabulário de pai, filhos e irmãos.
- O **DevTools** mostra o DOM (Elementos, ou Inspector no Firefox) e as requisições (Rede). Encaixado à direita, ele deixa a página visível enquanto você inspeciona.
- O código-fonte (`Ctrl + U`) mostra o HTML como chegou do servidor. O **JavaScript** pode alterar o DOM depois, e o `requests` não executa JavaScript.

**Boas práticas**

- O **robots.txt** declara o que pode ser acessado e em que ritmo. Ele é um pedido, e deve ser respeitado.
- Antes de coletar, confira se há API oficial, os termos de uso e a presença de dados pessoais (**LGPD**).

### Teste seus conhecimentos

Clique em cada pergunta para ver a resposta.

<details>
<summary><b>1.</b> Qual é a diferença entre o domínio e o endereço IP de um site?</summary>

O domínio é o **nome** (`books.toscrape.com`); o IP é o **número** do servidor na internet (`35.211.122.109`). O DNS faz a tradução de um para o outro.
</details>

<details>
<summary><b>2.</b> O seu scraper recebeu o código <code>429</code>. O que isso significa e o que fazer?</summary>

É um erro do cliente (`4xx`): **pedidos demais em pouco tempo**. O scraper deve diminuir o ritmo, fazendo pausas maiores entre as requisições.
</details>

<details>
<summary><b>3.</b> Em <code>https://loja.com/produtos?categoria=livros&page=3</code>, quais são os parâmetros e qual é o caminho?</summary>

O caminho é `/produtos`. Os parâmetros são `categoria=livros` e `page=3`.
</details>

<details>
<summary><b>4.</b> Uma informação aparece no painel de elementos do DevTools, mas não aparece no HTML baixado com o <code>requests</code>. Qual é a explicação mais provável?</summary>

A informação foi colocada no DOM por **JavaScript**, depois que a página chegou ao navegador. O `requests` só recebe o HTML original, sem executar os scripts. Para confirmar, use **Exibir código-fonte** (`Ctrl + U`).
</details>

<details>
<summary><b>5.</b> Por que o atributo <code>class</code> é tão importante para um scraper?</summary>

Porque elementos com o mesmo papel na página costumam ter a mesma classe (todos os preços com `price_color`, por exemplo). A classe funciona como um rótulo que permite encontrar todos esses elementos de uma vez.
</details>

### Glossário

| Termo | Significado |
|---|---|
| **Cliente** | Quem faz o pedido: o navegador ou um scraper |
| **Servidor** | O computador que guarda o site e responde aos pedidos |
| **IP** | O número que identifica um computador na rede |
| **DNS** | O serviço que traduz nomes de sites em endereços IP |
| **URL** | O endereço completo de um recurso: protocolo, domínio, porta, caminho e parâmetros |
| **Porta** | O número que indica qual programa do servidor vai atender o pedido |
| **HTTP / HTTPS** | O protocolo da web; o HTTPS é a versão criptografada |
| **Requisição** | A mensagem enviada pelo cliente |
| **Resposta** | A mensagem devolvida pelo servidor |
| **Método** | A ação pedida na requisição, como `GET` ou `POST` |
| **Código de status** | O número de três dígitos que resume o resultado do pedido |
| **Cabeçalho** | Uma informação extra sobre a mensagem, no formato `Nome: valor` |
| **HTML** | A linguagem que descreve a estrutura das páginas |
| **Tag / elemento** | A marcação do HTML (`<p>`) e o conjunto de abertura, conteúdo e fechamento |
| **Atributo** | Uma informação dentro da tag de abertura, como `class` ou `href` |
| **DOM** | A árvore de elementos que o navegador monta a partir do HTML |
| **DevTools** | As ferramentas do desenvolvedor do navegador |
| **robots.txt** | O arquivo com as regras de acesso para programas automáticos |

---

# Apêndice (opcional): um servidor web no seu computador

Este apêndice não é necessário para os módulos seguintes. Ele é para quem quer ver, na prática, o outro lado da conversa: o do **servidor**. O Python traz um servidor web simples, o `http.server`, que transforma uma pasta do computador em um site. Com ele, o seu computador passa a fazer os dois papéis ao mesmo tempo, o de servidor e o de cliente, e a relação entre caminho e arquivo da seção 1.3 aparece de forma literal:

<img src="imagens/url-pasta.png" alt="À esquerda, a pasta site-exemplo com seus arquivos; à direita, o endereço localhost correspondente a cada um; um caminho inexistente gera erro 404" width="1100">

### Passo 1: iniciar o servidor

O terminal onde o Jupyter está rodando continua ocupado, então abra um **segundo terminal**:

- **Jupyter no navegador:** abra uma nova janela do terminal, entre na pasta do curso e ative o `.venv`, como na rotina do Módulo 01.
- **VS Code:** use o menu **Terminal → New Terminal**.

Entre na pasta deste módulo e inicie o servidor:

⌨️
```
cd 02-fundamentos-da-web
python -m http.server 8000 --bind localhost --directory site-exemplo
```

O que cada parte do segundo comando faz:

| Parte | Significado |
|---|---|
| `python -m http.server` | Executa o servidor web que vem com o Python |
| `8000` | A **porta** em que o servidor vai atender |
| `--bind localhost` | Atende **apenas** pedidos vindos deste computador. Nenhum outro aparelho da rede consegue acessar os seus arquivos |
| `--directory site-exemplo` | A **pasta** cujos arquivos serão entregues |

<img src="imagens/terminal-servidor.png" alt="Terminal com o comando do servidor, a mensagem Serving HTTP e uma linha de registro para cada pedido recebido; abaixo, uma linha do registro explicada parte por parte" width="1100">

- ① O comando que inicia o servidor. O caminho da pasta é longo, e por isso o comando ocupa duas linhas.
- ② A confirmação de que o servidor está no ar. `::1` é o endereço do próprio computador na versão nova do protocolo IP (IPv6), equivalente ao `127.0.0.1`.
- ③ A partir daí, cada pedido recebido aparece como uma linha de **registro** (*log*). O pedido de `favicon.ico`, o ícone que aparece na aba do navegador, é feito automaticamente; como o site não tem esse arquivo, a resposta é 404, e isso não é um problema.

> ⚠️ Assim como o terminal do Jupyter, **este terminal precisa ficar aberto** enquanto o servidor estiver em uso.

### Passo 2: visitar o site pelo navegador

Abra uma nova aba do navegador e acesse os três endereços abaixo, observando o terminal do servidor a cada acesso:

<img src="imagens/site-local.png" alt="Três janelas de navegador: a página inicial da Livraria Exemplo, a listagem da pasta livros e a página de erro 404" width="1140">

- ① `http://localhost:8000/`: o caminho `/` entrega o arquivo `index.html`, a página inicial. O navegador pede também o `estilo.css`, que dá as cores e as fontes.
- ② `http://localhost:8000/livros/`: a pasta `livros` não tem um `index.html`, e por isso o servidor mostra a lista de arquivos que existem nela. Clique em um dos livros.
- ③ `http://localhost:8000/livros/nao-existe.html`: o arquivo não existe, e o servidor responde com o **erro 404**.

> ⚠️ Use sempre `localhost`, e não `127.0.0.1`: o servidor foi iniciado com `--bind localhost` e, dependendo do computador, pode não atender pelo número.

### Passo 3: acessar o servidor pelo Python

Agora o cliente passa a ser o Python. A célula abaixo pede quatro caminhos ao servidor local e mostra o código de status de cada resposta. Se o servidor não estiver no ar, ela avisa.

▶️ Execute:

In [ ]:
import requests

caminhos = ["/", "/livros", "/livros/iracema.html", "/livros/nao-existe.html"]

try:
    for caminho in caminhos:
        resposta = requests.get("http://localhost:8000" + caminho, allow_redirects=False, timeout=5)
        print(f"{caminho:26} {resposta.status_code} {resposta.reason}")
except requests.exceptions.ConnectionError:
    print("Não foi possível conectar ao servidor.")
    print("Confira se ele está rodando no outro terminal (Passo 1) e execute esta célula de novo.")

Cada pedido aparece também no registro do servidor, no outro terminal: para o servidor, o Python é só mais um cliente. O resultado esperado é:

- `/` e `/livros/iracema.html`: `200 OK`, os arquivos existem;
- `/livros/nao-existe.html`: `404 File not found`, o arquivo não existe;
- `/livros`, sem a barra no final: `301 Moved Permanently`. Como `livros` é uma pasta, o servidor indica que o endereço correto é `/livros/`, com a barra, o mesmo comportamento que você viu no scrapethissite na seção 2.3.

### Encerrando o servidor

Volte ao terminal em que o servidor está rodando e aperte **`Ctrl + C`**. O terminal volta a mostrar o prompt, e os endereços `localhost:8000` deixam de responder.

### Problemas comuns do apêndice

| Sintoma | Causa provável | Solução |
|---|---|---|
| Ao iniciar o servidor: `OSError: [WinError 10048]` ou `Address already in use` | Já existe um programa usando a porta 8000, talvez um servidor iniciado antes | Encerre o outro servidor (`Ctrl + C` no terminal dele) ou use outra porta, como `8001`, trocando também a porta nos endereços |
| A célula mostra "Não foi possível conectar ao servidor" | O servidor não está rodando | Inicie o servidor (Passo 1) no outro terminal e execute a célula de novo |
| O navegador não abre `http://127.0.0.1:8000` | O servidor está atendendo pelo endereço IPv6 do `localhost` | Use `http://localhost:8000` |
| No terminal: `python` não é reconhecido | O `.venv` não foi ativado neste terminal | Ative o ambiente, como na rotina do Módulo 01, ou use `py` no lugar de `python` |

> **Referência do apêndice:** [http.server](https://docs.python.org/pt-br/3/library/http.server.html) (documentação do Python).

---

## Próximo módulo

O **Módulo 03: requests** aprofunda a biblioteca usada neste módulo: como enviar parâmetros e cabeçalhos, definir um `User-Agent`, lidar com erros e com a codificação do texto (o caso do `Â£` da seção 2.4), e manter uma sessão. No final, você vai fazer a primeira coleta completa de dados de um site.

## Referências

Materiais consultados na elaboração deste módulo, agrupados pela parte em que foram usados. Os links também servem como leitura complementar.

**Referência principal da Parte 3 (HTML) e da seção 2.1 (requisição feita à mão)**

- SEVERANCE, Charles. *Django For Everybody: Full Python University Course*. freeCodeCamp.org, YouTube. Disponível em: [https://www.youtube.com/watch?v=o0XbHvKxw7Y](https://www.youtube.com/watch?v=o0XbHvKxw7Y). A explicação de HTML começa em [1:34:43](https://www.youtube.com/watch?v=o0XbHvKxw7Y&t=5683s); o início do curso trata do ciclo de requisição e resposta e da conversa com o servidor por meio de *sockets*. Em inglês. Veja também a [apresentação do curso no blog do freeCodeCamp](https://www.freecodecamp.org/news/django-for-everybody-learn-the-popular-python-framework-from-dr-chuck/).

**Parte 1: como a web funciona**

- MDN Web Docs: [Como a web funciona](https://developer.mozilla.org/pt-BR/docs/Learn_web_development/Getting_started/Web_standards/How_the_web_works); [What is a URL?](https://developer.mozilla.org/en-US/docs/Learn_web_development/Howto/Web_mechanics/What_is_a_URL) e [What is a domain name?](https://developer.mozilla.org/en-US/docs/Learn_web_development/Howto/Web_mechanics/What_is_a_domain_name) (em inglês).
- Documentação do Python: [urllib.parse](https://docs.python.org/pt-br/3/library/urllib.parse.html) e [socket](https://docs.python.org/pt-br/3/library/socket.html).

**Parte 2: HTTP**

- MDN Web Docs: [Uma visão geral do HTTP](https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Guides/Overview), [Mensagens HTTP](https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Guides/Messages), [Métodos de requisição HTTP](https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Reference/Methods), [Códigos de status de respostas HTTP](https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Reference/Status) e [Cabeçalhos HTTP](https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Reference/Headers).
- IETF. [RFC 9110: HTTP Semantics](https://www.rfc-editor.org/rfc/rfc9110), 2022. A especificação oficial do HTTP, em inglês.

**Parte 3: HTML, DOM e DevTools**

- MDN Web Docs: [HTML básico](https://developer.mozilla.org/pt-BR/docs/Learn_web_development/Getting_started/Your_first_website/Creating_the_content), [Iniciando com HTML](https://developer.mozilla.org/pt-BR/docs/Learn_web_development/Core/Structuring_content/Basic_HTML_syntax) e [Modelo de Objeto de Documento (DOM)](https://developer.mozilla.org/pt-BR/docs/Web/API/Document_Object_Model).
- Mozilla: [Firefox DevTools User Docs](https://firefox-source-docs.mozilla.org/devtools-user/), em especial [Page Inspector](https://firefox-source-docs.mozilla.org/devtools-user/page_inspector/), [Network Monitor](https://firefox-source-docs.mozilla.org/devtools-user/network_monitor/) e [Toolbox](https://firefox-source-docs.mozilla.org/devtools-user/tools_toolbox/) (posições do painel). Em inglês.
- Microsoft: [Visão geral do DevTools](https://learn.microsoft.com/pt-br/microsoft-edge/devtools/overview), do Edge. Google: [Chrome DevTools](https://developer.chrome.com/docs/devtools?hl=pt-br).

**Parte 4: boas práticas**

- Google: [Introdução ao robots.txt](https://developers.google.com/search/docs/crawling-indexing/robots/intro?hl=pt-br).
- IETF. [RFC 9309: Robots Exclusion Protocol](https://www.rfc-editor.org/rfc/rfc9309), 2022. Em inglês.
- Documentação do Python: [urllib.robotparser](https://docs.python.org/pt-br/3/library/urllib.robotparser.html).
- BRASIL. [Lei nº 13.709, de 14 de agosto de 2018](https://www.planalto.gov.br/ccivil_03/_ato2015-2018/2018/lei/l13709.htm) (Lei Geral de Proteção de Dados); [Autoridade Nacional de Proteção de Dados (ANPD)](https://www.gov.br/anpd/pt-br).

**Apêndice**

- Documentação do Python: [http.server](https://docs.python.org/pt-br/3/library/http.server.html).

**Sites de treino usados nos exemplos:** [books.toscrape.com](https://books.toscrape.com), [quotes.toscrape.com](https://quotes.toscrape.com), [scrapethissite.com](https://www.scrapethissite.com/pages/) e [webscraper.io/test-sites](https://webscraper.io/test-sites).